# Advanced Tutorial: Python Set Update Operations

This notebook develops advanced problem-solving skills with **mutating set operations**:

- `set.update(...)` and `|=`
- `set.intersection_update(...)` and `&=`
- `set.difference_update(...)` and `-=`
- `set.symmetric_difference_update(...)` and `^=`

The emphasis is not only on obtaining the correct elements.

We will also study:

- mutation versus rebinding,
- shared references and aliases,
- processing streamed or paginated data,
- preserving invariants,
- applying changes safely,
- validating input before mutation,
- using sets in graph, permission, inventory, and synchronization problems.

Each problem is broken into logical steps:

1. understand the data,
2. predict the transformation,
3. choose the correct update operation,
4. implement the mutation,
5. verify identity and content,
6. discuss design trade-offs.


## 1. Core Review: New Set Versus In-Place Update

A normal set expression creates a new set:

```python
result = left | right
```

An update operation changes an existing set:

```python
left |= right
```

The distinction matters whenever another variable, object, or function holds a reference to the original set.


In [1]:
left = {1, 2, 3}
right = {3, 4, 5}

original_id = id(left)
alias = left

left = left | right

print("left:", left)
print("alias:", alias)
print("same object:", left is alias)
print("original id preserved:", id(left) == original_id)


left: {1, 2, 3, 4, 5}
alias: {1, 2, 3}
same object: False
original id preserved: False


The assignment above **rebinds** `left`.

The variable `alias` still points to the old object.

Now compare that behavior with an in-place update.


In [2]:
left = {1, 2, 3}
right = {3, 4, 5}

original_id = id(left)
alias = left

left |= right

print("left:", left)
print("alias:", alias)
print("same object:", left is alias)
print("original id preserved:", id(left) == original_id)

assert left == {1, 2, 3, 4, 5}
assert alias == left
assert left is alias
assert id(left) == original_id


left: {1, 2, 3, 4, 5}
alias: {1, 2, 3, 4, 5}
same object: True
original id preserved: True


### Best-practice rule

Use an update operation when preserving the identity of the original set is part of the function's contract.

Use a non-mutating expression when returning a separate result is clearer or safer.


## 2. Quick Reference

| Goal | Method | Operator | Mutates target? |
|---|---|---|---|
| Add all elements | `s.update(items)` | `s |= other_set` | Yes |
| Keep common elements | `s.intersection_update(items)` | `s &= other_set` | Yes |
| Remove matching elements | `s.difference_update(items)` | `s -= other_set` | Yes |
| Toggle membership | `s.symmetric_difference_update(items)` | `s ^= other_set` | Yes |

Methods accept general iterables.

The binary operators normally expect set-like operands.


# Problem 1: Shared Live Subscription Registry

A service stores active subscription topics in a set.

Several components hold the **same set reference**:

- a dashboard,
- a notification worker,
- a monitoring function.

A configuration refresh must add newly enabled topics and remove deprecated topics **without replacing the original set object**.


### Step 1: Examine the starting state

We have:

- current topics,
- topics enabled by the new configuration,
- topics that must be removed.

The final result should contain every current or newly enabled topic except deprecated ones.


In [3]:
subscriptions = {"orders", "payments", "inventory"}
dashboard_view = subscriptions
worker_view = subscriptions

enabled = {"shipping", "returns", "payments"}
deprecated = {"inventory", "legacy_alerts"}

print("subscriptions:", subscriptions)
print("dashboard_view is subscriptions:", dashboard_view is subscriptions)
print("worker_view is subscriptions:", worker_view is subscriptions)


subscriptions: {'orders', 'payments', 'inventory'}
dashboard_view is subscriptions: True
worker_view is subscriptions: True


### Step 2: Predict the two mutations

First perform a union update:

```python
subscriptions.update(enabled)
```

Then remove deprecated topics:

```python
subscriptions.difference_update(deprecated)
```

Before running the solution, predict the final set.


### Solution


In [4]:
original_id = id(subscriptions)

subscriptions.update(enabled)
subscriptions.difference_update(deprecated)

expected = {"orders", "payments", "shipping", "returns"}

print("updated subscriptions:", subscriptions)
print("dashboard sees:", dashboard_view)
print("worker sees:", worker_view)
print("identity preserved:", id(subscriptions) == original_id)

assert subscriptions == expected
assert dashboard_view == expected
assert worker_view == expected
assert subscriptions is dashboard_view is worker_view
assert id(subscriptions) == original_id


updated subscriptions: {'orders', 'shipping', 'returns', 'payments'}
dashboard sees: {'orders', 'shipping', 'returns', 'payments'}
worker sees: {'orders', 'shipping', 'returns', 'payments'}
identity preserved: True


### Discussion

This is a strong use case for mutation.

Rebinding `subscriptions` would not update the references already stored by the dashboard and worker.

The order also matters conceptually:

1. add all enabled topics,
2. remove everything forbidden.

This guarantees that a deprecated topic stays absent even if it accidentally appears in `enabled`.


# Problem 2: Incremental Access-Control Reconciliation

A user currently has a set of permissions.

A policy engine provides:

- permissions granted by the new role,
- permissions revoked explicitly,
- permissions allowed by the environment.

The final permission set must:

1. include role grants,
2. exclude explicit revocations,
3. contain only environment-allowed permissions.

The original set must be mutated.


### Step 1: Translate each business rule into a set update

- Include role grants → union update.
- Exclude revocations → difference update.
- Keep only allowed permissions → intersection update.


In [5]:
permissions = {"read_reports", "export_data", "edit_users"}
role_grants = {"read_reports", "manage_billing", "view_audit"}
revoked = {"edit_users", "export_data"}
environment_allowed = {
    "read_reports",
    "manage_billing",
    "view_audit",
    "use_sandbox",
}

original_id = id(permissions)


### Step 2: Apply the operations in policy order

Policy order is important.

Explicit revocations should be applied after grants, so a revoked permission cannot be restored by the same refresh.


In [6]:
permissions.update(role_grants)
permissions.difference_update(revoked)
permissions.intersection_update(environment_allowed)

print("effective permissions:", permissions)

expected = {"read_reports", "manage_billing", "view_audit"}

assert permissions == expected
assert id(permissions) == original_id


effective permissions: {'view_audit', 'read_reports', 'manage_billing'}


### Step 3: Package the logic as a reusable function


In [7]:
def reconcile_permissions(
    current,
    *,
    grants,
    revoked,
    allowed,
):
    """Mutate current permissions according to policy constraints."""
    current.update(grants)
    current.difference_update(revoked)
    current.intersection_update(allowed)


current = {"read", "write", "delete"}
same_reference = current

reconcile_permissions(
    current,
    grants={"share", "archive"},
    revoked={"delete", "archive"},
    allowed={"read", "write", "share"},
)

print(current)

assert current == {"read", "write", "share"}
assert current is same_reference


{'share', 'write', 'read'}


### Best-practice note

Keyword-only policy arguments make accidental argument-order mistakes less likely.

The function name and docstring should clearly state that the first argument is mutated.


# Problem 3: Streaming Deduplication with an Ignore List

A data source yields pages of identifiers.

We want to maintain one set containing all unique identifiers, while permanently excluding blocked identifiers.

The set must be updated page by page.


### Step 1: Create a simulated paginated source


In [8]:
def read_identifier_pages():
    yield ["A-100", "A-101", "A-102", "BOT-1"]
    yield ["A-102", "A-103", "BOT-2", "A-104"]
    yield ["A-100", "A-105", "BOT-1", "A-106"]


### Step 2: Choose an invariant

After every loop iteration:

> `seen` contains every valid identifier encountered so far and no blocked identifier.

Maintaining an invariant after every page is often safer than cleaning only at the end.


In [9]:
blocked = {"BOT-1", "BOT-2", "TEST"}
seen = set()

for page_number, page in enumerate(read_identifier_pages(), start=1):
    seen.update(page)
    seen.difference_update(blocked)

    print(f"after page {page_number}:", sorted(seen))
    assert seen.isdisjoint(blocked)

expected = {"A-100", "A-101", "A-102", "A-103", "A-104", "A-105", "A-106"}
assert seen == expected


after page 1: ['A-100', 'A-101', 'A-102']
after page 2: ['A-100', 'A-101', 'A-102', 'A-103', 'A-104']
after page 3: ['A-100', 'A-101', 'A-102', 'A-103', 'A-104', 'A-105', 'A-106']


### Step 3: Refactor the invariant into a function


In [10]:
def absorb_page(target, page, *, blocked=()):
    """Add page values to target, then remove blocked values in place."""
    target.update(page)
    target.difference_update(blocked)


collected = set()

for page in read_identifier_pages():
    absorb_page(collected, page, blocked=blocked)

assert collected == expected
print(collected)


{'A-102', 'A-101', 'A-103', 'A-100', 'A-105', 'A-104', 'A-106'}


### Discussion

The method form is especially convenient here because `page` is a list.

With operators, we would need an explicit conversion such as:

```python
seen |= set(page)
```

`update(page)` is clearer and avoids creating that temporary set explicitly.


# Problem 4: Safe Mutation with Pre-Validation

A set stores registered feature names.

An incoming iterable may contain invalid values such as lists or dictionaries, which are unhashable.

A naive call to `set.update(...)` can mutate the target partially before an exception interrupts processing.

We want **all-or-nothing behavior**.


### Step 1: Demonstrate the risk

The exact partial state can depend on iteration order.

The important point is that mutation may begin before the invalid value is encountered.


In [11]:
registry = {"stable"}

incoming = ["alpha", "beta", ["not", "hashable"], "gamma"]

try:
    registry.update(incoming)
except TypeError as exc:
    print("error:", exc)

print("registry after failed update:", registry)


error: unhashable type: 'list'
registry after failed update: {'beta', 'stable', 'alpha'}


### Step 2: Build a temporary validated set

Constructing `set(incoming)` completes validation before the target is touched.

Only after validation succeeds do we mutate the original registry.


In [12]:
def atomic_update(target, incoming):
    """Update target only if every incoming element is hashable."""
    validated = set(incoming)
    target.update(validated)


registry = {"stable"}
alias = registry

try:
    atomic_update(registry, ["alpha", "beta", ["bad"], "gamma"])
except TypeError:
    pass

print("unchanged after invalid batch:", registry)

assert registry == {"stable"}
assert registry is alias


unchanged after invalid batch: {'stable'}


### Step 3: Verify a successful batch


In [13]:
atomic_update(registry, ["alpha", "beta", "alpha", "gamma"])

print(registry)

assert registry == {"stable", "alpha", "beta", "gamma"}
assert registry is alias


{'beta', 'stable', 'alpha', 'gamma'}


### Discussion

This pattern uses temporary memory to gain transactional behavior.

It is a useful design when:

- partial mutation would corrupt state,
- incoming data is not trusted,
- validation errors must leave the target unchanged.


# Problem 5: Detecting Membership Changes with Symmetric Difference

Two snapshots contain identifiers from consecutive system scans.

We want the identifiers whose membership changed:

- newly appeared identifiers,
- disappeared identifiers.

That is exactly the symmetric difference.


In [14]:
previous = {"node-1", "node-2", "node-3", "node-5"}
current = {"node-2", "node-3", "node-4", "node-6"}

changed = previous ^ current

print("changed membership:", changed)

assert changed == {"node-1", "node-4", "node-5", "node-6"}


changed membership: {'node-6', 'node-4', 'node-5', 'node-1'}


### Step 1: Separate additions and removals

Symmetric difference tells us *what changed*, but not the direction.

Use two differences to classify the change.


In [15]:
added = current - previous
removed = previous - current

print("added:", added)
print("removed:", removed)

assert added == {"node-4", "node-6"}
assert removed == {"node-1", "node-5"}
assert changed == added | removed


added: {'node-4', 'node-6'}
removed: {'node-1', 'node-5'}


### Step 2: Mutate a reusable change buffer

Suppose a monitoring object owns a set named `change_buffer`, and external code keeps a reference to it.

We can overwrite its logical content without replacing the object:

1. clear it,
2. update it with the previous snapshot,
3. apply a symmetric-difference update with the current snapshot.


In [16]:
change_buffer = {"stale-placeholder"}
external_view = change_buffer
original_id = id(change_buffer)

change_buffer.clear()
change_buffer.update(previous)
change_buffer.symmetric_difference_update(current)

print(change_buffer)

assert change_buffer == changed
assert change_buffer is external_view
assert id(change_buffer) == original_id


{'node-6', 'node-4', 'node-5', 'node-1'}


### Discussion

`clear()` followed by update operations is a common pattern when an object must retain identity but its entire logical content must be recomputed.


# Problem 6: Difference Is Not Associative

Consider three sets:

- `base`
- `remove_first`
- `restore_candidates`

Compare:

```python
base - (remove_first - restore_candidates)
```

with:

```python
(base - remove_first) - restore_candidates
```

These expressions have different meanings.


In [17]:
base = {"A", "B", "C", "D"}
remove_first = {"B", "C"}
restore_candidates = {"C", "D"}

expression_1 = base - (remove_first - restore_candidates)
expression_2 = (base - remove_first) - restore_candidates

print("base - (remove_first - restore_candidates):", expression_1)
print("(base - remove_first) - restore_candidates:", expression_2)

assert expression_1 == {"A", "C", "D"}
assert expression_2 == {"A"}
assert expression_1 != expression_2


base - (remove_first - restore_candidates): {'A', 'D', 'C'}
(base - remove_first) - restore_candidates: {'A'}


### Step 1: Reproduce the second expression by mutation

`difference_update(remove_first, restore_candidates)` removes every element found in either iterable.

It corresponds to:

```python
base - remove_first - restore_candidates
```

not to:

```python
base - (remove_first - restore_candidates)
```


In [18]:
working = base.copy()
working.difference_update(remove_first, restore_candidates)

print(working)

assert working == expression_2


{'A'}


### Step 2: Reproduce the first expression safely

Compute the nested difference first, then remove that result.


In [19]:
working = base.copy()
effective_removals = remove_first - restore_candidates
working.difference_update(effective_removals)

print("effective removals:", effective_removals)
print("result:", working)

assert working == expression_1


effective removals: {'B'}
result: {'A', 'D', 'C'}


### Best-practice note

Use intermediate names when set expressions encode business meaning.

A name such as `effective_removals` is easier to review than a dense nested expression.


# Problem 7: Maintaining an Eligible Candidate Pool

A hiring pipeline tracks candidates still eligible for a specialized interview.

A candidate must:

- appear in the applicant pool,
- have passed the technical screen,
- have work authorization,
- not have withdrawn,
- not be marked duplicate.

We will mutate one candidate set through successive constraints.


In [20]:
eligible = {"Ana", "Boris", "Chen", "Dina", "Eli", "Fatima"}
technical_pass = {"Ana", "Chen", "Dina", "Fatima"}
authorized = {"Ana", "Boris", "Chen", "Fatima"}
withdrawn = {"Fatima"}
duplicates = {"Chen", "Ghost Record"}

original_id = id(eligible)


### Step 1: Apply positive constraints with intersection updates

A candidate must satisfy both positive requirements.


In [21]:
eligible.intersection_update(technical_pass)
print("after technical screen:", eligible)

eligible.intersection_update(authorized)
print("after authorization:", eligible)


after technical screen: {'Fatima', 'Ana', 'Chen', 'Dina'}
after authorization: {'Fatima', 'Ana', 'Chen'}


### Step 2: Apply negative constraints with difference updates


In [22]:
eligible.difference_update(withdrawn, duplicates)

print("final eligible candidates:", eligible)

assert eligible == {"Ana"}
assert id(eligible) == original_id


final eligible candidates: {'Ana'}


### Step 3: Generalize to any number of positive and negative filters


In [23]:
def filter_pool_in_place(pool, *, required_groups=(), excluded_groups=()):
    """Mutate pool so it satisfies all required groups and no excluded group."""
    for group in required_groups:
        pool.intersection_update(group)

    for group in excluded_groups:
        pool.difference_update(group)


pool = {"A", "B", "C", "D", "E"}

filter_pool_in_place(
    pool,
    required_groups=[
        {"A", "B", "C", "D"},
        {"B", "C", "D", "E"},
        {"C", "D"},
    ],
    excluded_groups=[
        {"D"},
    ],
)

assert pool == {"C"}
print(pool)


{'C'}


### Discussion

Repeated `intersection_update` operations naturally model “must satisfy every rule.”

Repeated `difference_update` operations naturally model “must satisfy none of these exclusions.”


# Problem 8: Inventory Synchronization with Protected Items

A local cache contains product codes.

A remote snapshot is authoritative, except that locally protected codes must never be removed.

We want to mutate the cache so that it contains:

```text
remote snapshot ∪ protected local items
```

while preserving the original cache object.


In [24]:
cache = {"P100", "P200", "P300", "LOCAL-DEMO"}
remote_snapshot = {"P200", "P300", "P400", "P500"}
protected = {"LOCAL-DEMO"}

alias = cache
original_id = id(cache)


### Step 1: Capture the protected items currently present

The protected set may contain codes that are not actually in the cache.

Only protected items currently stored locally should be retained.


In [25]:
protected_present = cache & protected

print(protected_present)

assert protected_present == {"LOCAL-DEMO"}


{'LOCAL-DEMO'}


### Step 2: Replace the logical contents without rebinding

We can:

1. clear the cache,
2. update from the remote snapshot,
3. restore protected local entries.


In [26]:
cache.clear()
cache.update(remote_snapshot)
cache.update(protected_present)

print(cache)

expected = {"P200", "P300", "P400", "P500", "LOCAL-DEMO"}

assert cache == expected
assert cache is alias
assert id(cache) == original_id


{'LOCAL-DEMO', 'P200', 'P500', 'P300', 'P400'}


### Alternative formulation

We could first remove everything not in the remote snapshot, then add remote values and protected items.

However, `clear()` plus explicit updates often communicates “replace logical contents” more directly.


# Problem 9: Graph Frontier Expansion

A graph traversal maintains:

- `visited`: nodes already processed,
- `frontier`: nodes to process next.

For each layer:

1. mark frontier nodes as visited,
2. collect all neighbors,
3. remove already visited nodes,
4. mutate the existing frontier set to hold the next layer.

This is a realistic use of several set update operations together.


In [27]:
graph = {
    "A": {"B", "C"},
    "B": {"A", "D", "E"},
    "C": {"A", "F"},
    "D": {"B"},
    "E": {"B", "F"},
    "F": {"C", "E", "G"},
    "G": {"F"},
}

visited = set()
frontier = {"A"}
frontier_alias = frontier


### Step 1: Expand one layer manually


In [28]:
visited.update(frontier)

neighbors = set()
for node in frontier:
    neighbors.update(graph[node])

neighbors.difference_update(visited)

frontier.clear()
frontier.update(neighbors)

print("visited:", visited)
print("next frontier:", frontier)

assert visited == {"A"}
assert frontier == {"B", "C"}
assert frontier is frontier_alias


visited: {'A'}
next frontier: {'B', 'C'}


### Step 2: Traverse all reachable nodes

We continue until the frontier becomes empty.


In [29]:
layers = [{"A"}]
visited = set()
frontier = {"A"}
frontier_alias = frontier

while frontier:
    visited.update(frontier)

    next_nodes = set()
    for node in frontier:
        next_nodes.update(graph[node])

    next_nodes.difference_update(visited)

    frontier.clear()
    frontier.update(next_nodes)

    if frontier:
        layers.append(frontier.copy())

print("layers:", layers)
print("visited:", visited)

assert layers == [{"A"}, {"B", "C"}, {"D", "E", "F"}, {"G"}]
assert visited == set(graph)
assert frontier == set()
assert frontier is frontier_alias


layers: [{'A'}, {'B', 'C'}, {'D', 'E', 'F'}, {'G'}]
visited: {'G', 'C', 'A', 'F', 'B', 'D', 'E'}


### Discussion

Notice the deliberate copy in:

```python
layers.append(frontier.copy())
```

Appending `frontier` directly would store multiple references to the same mutable object.

After the traversal, every recorded layer would appear empty.


# Problem 10: Toggle-Based Feature Flags

A symmetric-difference update toggles membership:

- present elements are removed,
- absent elements are added.

This can model a batch of feature-flag toggles.


In [30]:
enabled_flags = {"dark_mode", "beta_search", "new_checkout"}
toggle_request = {"beta_search", "compact_nav", "new_checkout", "ai_help"}

original_id = id(enabled_flags)

enabled_flags.symmetric_difference_update(toggle_request)

print(enabled_flags)

expected = {"dark_mode", "compact_nav", "ai_help"}

assert enabled_flags == expected
assert id(enabled_flags) == original_id


{'ai_help', 'compact_nav', 'dark_mode'}


### Step 1: Understand duplicate toggle requests

A set cannot represent duplicate requests.

If the original event stream is:

```python
["A", "A", "B"]
```

converting it to a set produces `{"A", "B"}`.

But applying toggles sequentially would toggle `A` twice, leaving it unchanged.

Therefore, a set is only correct when each feature appears at most once per batch, or when duplicates should be ignored.


In [31]:
events = ["A", "A", "B"]

sequential = set()
for event in events:
    sequential.symmetric_difference_update({event})

batched = set()
batched.symmetric_difference_update(set(events))

print("sequential result:", sequential)
print("deduplicated batch result:", batched)

assert sequential == {"B"}
assert batched == {"A", "B"}


sequential result: {'B'}
deduplicated batch result: {'A', 'B'}


### Best-practice note

Before using a set-based batch toggle, define the semantics of duplicate events.

Data structure choice is part of the problem specification.


# Problem 11: Rollback After a Failed In-Place Transformation

Sometimes mutation is required, but a later validation rule may reject the final state.

We want to restore the original contents **without replacing the target object**.


### Step 1: Save a snapshot

A shallow copy is sufficient when set elements are immutable identifiers.


In [32]:
active_regions = {"eu-west", "us-east", "ap-south"}
alias = active_regions
snapshot = active_regions.copy()
original_id = id(active_regions)


### Step 2: Apply a risky mutation


In [33]:
active_regions.update({"test-lab", "moon-base"})
active_regions.difference_update({"us-east"})

print("candidate state:", active_regions)


candidate state: {'moon-base', 'test-lab', 'eu-west', 'ap-south'}


### Step 3: Validate and roll back in place

Suppose every region must start with one of the approved prefixes.


In [34]:
approved_prefixes = ("eu-", "us-", "ap-")

is_valid = all(region.startswith(approved_prefixes) for region in active_regions)

if not is_valid:
    active_regions.clear()
    active_regions.update(snapshot)

print("final state:", active_regions)

assert active_regions == {"eu-west", "us-east", "ap-south"}
assert active_regions is alias
assert id(active_regions) == original_id


final state: {'eu-west', 'us-east', 'ap-south'}


### Discussion

Rollback requires both:

- a snapshot of the old logical content,
- in-place restoration using `clear()` and `update()`.

Rebinding with `active_regions = snapshot` would break aliases.


# Problem 12: Reconciliation Report Plus In-Place Update

A synchronization function should:

1. calculate what will be added,
2. calculate what will be removed,
3. mutate the local set to match the remote set,
4. return a report.

This combines non-mutating analysis with mutating application.


In [35]:
def reconcile_exact(target, desired):
    """Mutate target to equal desired and return added/removed sets."""
    desired_set = set(desired)

    added = desired_set - target
    removed = target - desired_set

    target.intersection_update(desired_set)
    target.update(desired_set)

    return {
        "added": added,
        "removed": removed,
    }


local = {"A", "B", "C", "OLD"}
remote = ["B", "C", "D", "E", "E"]

alias = local
report = reconcile_exact(local, remote)

print("local:", local)
print("report:", report)

assert local == {"B", "C", "D", "E"}
assert report == {
    "added": {"D", "E"},
    "removed": {"A", "OLD"},
}
assert local is alias


local: {'C', 'B', 'D', 'E'}
report: {'added': {'D', 'E'}, 'removed': {'A', 'OLD'}}


### Why use both intersection and update?

After:

```python
target.intersection_update(desired_set)
```

all obsolete values are gone.

After:

```python
target.update(desired_set)
```

all missing desired values are present.

The result exactly matches `desired_set`, while preserving target identity.


# Problem 13: Multi-Tenant Data Visibility

A user may view records satisfying all of these rules:

- the record belongs to one of the user's tenants,
- the record appears in the current search result,
- the record is not confidential unless explicitly cleared,
- the record is not archived.

We represent each category as a set of record IDs.


In [36]:
visible = {"R1", "R2", "R3", "R4", "R5", "R6"}

tenant_records = {"R1", "R2", "R3", "R5", "R7"}
search_result = {"R2", "R3", "R4", "R5", "R8"}
confidential = {"R3", "R6"}
confidential_clearance = {"R3"}
archived = {"R5", "R9"}


### Step 1: Apply positive visibility filters


In [37]:
visible.intersection_update(tenant_records)
visible.intersection_update(search_result)

print("after positive filters:", visible)

assert visible == {"R2", "R3", "R5"}


after positive filters: {'R3', 'R5', 'R2'}


### Step 2: Compute confidential records that remain forbidden

A confidential record is forbidden only when it is not in the clearance set.


In [38]:
forbidden_confidential = confidential - confidential_clearance

print(forbidden_confidential)

assert forbidden_confidential == {"R6"}


{'R6'}


### Step 3: Remove all negative categories


In [39]:
visible.difference_update(forbidden_confidential, archived)

print("final visible records:", visible)

assert visible == {"R2", "R3"}


final visible records: {'R3', 'R2'}


### Discussion

This problem illustrates a useful pattern:

1. derive a meaningful intermediate set,
2. use that set in an in-place update.

Trying to compress every rule into one expression usually reduces readability.


# Problem 14: Mutation Contracts and Defensive Copying

A function receives a caller-owned set.

There are two reasonable API designs:

1. mutate the caller's set,
2. return a new transformed set.

Both are valid, but the contract must be explicit.


In [40]:
def normalize_tags_in_place(tags, *, allowed, required=()):
    """Mutate tags so they are allowed, then add required tags."""
    tags.intersection_update(allowed)
    tags.update(required)


def normalized_tags(tags, *, allowed, required=()):
    """Return a normalized copy and leave the input unchanged."""
    result = set(tags)
    result.intersection_update(allowed)
    result.update(required)
    return result


### Step 1: Compare caller-visible behavior


In [41]:
allowed = {"python", "data", "backend", "featured"}

original = {"python", "backend", "obsolete"}
alias = original

normalize_tags_in_place(
    original,
    allowed=allowed,
    required={"featured"},
)

print("mutated original:", original)
assert original == {"python", "backend", "featured"}
assert original is alias


mutated original: {'python', 'featured', 'backend'}


In [42]:
original = {"python", "backend", "obsolete"}
alias = original

result = normalized_tags(
    original,
    allowed=allowed,
    required={"featured"},
)

print("original:", original)
print("result:", result)

assert original == {"python", "backend", "obsolete"}
assert result == {"python", "backend", "featured"}
assert original is alias
assert result is not original


original: {'obsolete', 'python', 'backend'}
result: {'python', 'featured', 'backend'}


### Best-practice checklist for mutating functions

A mutating function should usually:

- use a name or docstring that communicates mutation,
- avoid returning the same set unless method chaining is intentional,
- validate risky input before mutation,
- preserve documented invariants,
- include tests for both content and object identity.


# Problem 15: Advanced Composite Challenge — Live Fraud Rules

A fraud-detection service maintains a live set of blocked account IDs.

Every refresh supplies:

- newly blocked accounts,
- accounts cleared by investigators,
- accounts still recognized by the current data source,
- emergency overrides that must always stay blocked.

Requirements:

1. mutate the existing set,
2. add newly blocked accounts,
3. remove cleared accounts,
4. discard accounts no longer recognized,
5. restore emergency overrides,
6. preserve aliases,
7. return a change report relative to the original state.


### Step 1: Prepare the data


In [43]:
blocked_accounts = {"A10", "A20", "A30", "LEGACY-X"}
dashboard_reference = blocked_accounts

newly_blocked = {"A40", "A50", "A20"}
cleared = {"A30", "A50"}
recognized = {"A10", "A20", "A30", "A40", "A50", "A60"}
emergency_overrides = {"VIP-FRAUD", "A50"}


### Step 2: Design the operation order

A suitable order is:

1. snapshot the original state,
2. add newly blocked accounts,
3. remove cleared accounts,
4. retain only recognized accounts,
5. add emergency overrides last.

Emergency overrides come last because they take precedence over clearing and recognition rules.


### Step 3: Implement the solution


In [44]:
def refresh_blocked_accounts(
    target,
    *,
    newly_blocked,
    cleared,
    recognized,
    emergency_overrides,
):
    """Mutate target according to precedence rules and return a change report."""
    before = target.copy()

    target.update(newly_blocked)
    target.difference_update(cleared)
    target.intersection_update(recognized)
    target.update(emergency_overrides)

    return {
        "added": target - before,
        "removed": before - target,
        "unchanged": before & target,
    }


original_id = id(blocked_accounts)

report = refresh_blocked_accounts(
    blocked_accounts,
    newly_blocked=newly_blocked,
    cleared=cleared,
    recognized=recognized,
    emergency_overrides=emergency_overrides,
)

print("blocked accounts:", blocked_accounts)
print("report:", report)


blocked accounts: {'A50', 'A40', 'A20', 'A10', 'VIP-FRAUD'}
report: {'added': {'A50', 'A40', 'VIP-FRAUD'}, 'removed': {'LEGACY-X', 'A30'}, 'unchanged': {'A20', 'A10'}}


### Step 4: Verify every requirement


In [45]:
expected_blocked = {"A10", "A20", "A40", "A50", "VIP-FRAUD"}

assert blocked_accounts == expected_blocked
assert blocked_accounts is dashboard_reference
assert id(blocked_accounts) == original_id

assert report["added"] == {"A40", "A50", "VIP-FRAUD"}
assert report["removed"] == {"A30", "LEGACY-X"}
assert report["unchanged"] == {"A10", "A20"}

assert (
    report["added"]
    | report["removed"]
    | report["unchanged"]
) == {
    "A10",
    "A20",
    "A30",
    "A40",
    "A50",
    "LEGACY-X",
    "VIP-FRAUD",
}

print("all requirements verified")


all requirements verified


### Discussion

This composite problem uses all four major ideas:

- union updates for additions and overrides,
- difference updates for explicit removals,
- intersection updates for authoritative filtering,
- non-mutating set expressions for reporting.

The operation order encodes policy precedence.


# Common Pitfalls


## Pitfall 1: Expecting an update method to return the set

Mutation methods return `None`.


In [46]:
values = {1, 2}
result = values.update([2, 3, 4])

print("values:", values)
print("return value:", result)

assert values == {1, 2, 3, 4}
assert result is None


values: {1, 2, 3, 4}
return value: None


Do not write:

```python
values = values.update(other)
```

That would replace `values` with `None`.


## Pitfall 2: Mutating a set while iterating over it


In [47]:
values = {1, 2, 3, 4, 5}

try:
    for value in values:
        if value % 2 == 0:
            values.remove(value)
except RuntimeError as exc:
    print(type(exc).__name__ + ":", exc)


RuntimeError: Set changed size during iteration


Use a separate derived set, then perform one update.


In [48]:
values = {1, 2, 3, 4, 5}

to_remove = {value for value in values if value % 2 == 0}
values.difference_update(to_remove)

assert values == {1, 3, 5}
print(values)


{1, 3, 5}


## Pitfall 3: Assuming printed set order is meaningful

Sets are unordered collections.

Tests should compare sets directly rather than compare their printed representation.


In [49]:
actual = {"beta", "alpha", "gamma"}
expected = {"gamma", "beta", "alpha"}

assert actual == expected

print("stable display for teaching:", sorted(actual))


stable display for teaching: ['alpha', 'beta', 'gamma']


## Pitfall 4: Losing alias visibility through rebinding


In [50]:
target = {"A", "B"}
alias = target

target = target | {"C"}

assert target == {"A", "B", "C"}
assert alias == {"A", "B"}
assert target is not alias


When aliases must observe the update, mutate:


In [51]:
target = {"A", "B"}
alias = target

target.update({"C"})

assert target == {"A", "B", "C"}
assert alias == {"A", "B", "C"}
assert target is alias


# Final Practice Problems

Try each problem before revealing the solution cell.


## Practice A: Active Experiment Cohort

You have:

```python
cohort = {"U1", "U2", "U3", "U4"}
qualified = {"U2", "U3", "U4", "U5"}
opted_out = {"U3"}
forced_in = {"ADMIN"}
```

Mutate `cohort` so it contains qualified users, excludes opt-outs, and includes forced users.

Preserve object identity.


In [52]:
cohort = {"U1", "U2", "U3", "U4"}
alias = cohort
original_id = id(cohort)

qualified = {"U2", "U3", "U4", "U5"}
opted_out = {"U3"}
forced_in = {"ADMIN"}

# Solution
cohort.intersection_update(qualified)
cohort.difference_update(opted_out)
cohort.update(forced_in)

assert cohort == {"U2", "U4", "ADMIN"}
assert cohort is alias
assert id(cohort) == original_id

print(cohort)


{'U2', 'U4', 'ADMIN'}


## Practice B: Changed Configuration Keys

Given two configuration-key snapshots, mutate a reusable set named `changed_keys` so it contains keys appearing in exactly one snapshot.


In [53]:
old_keys = {"host", "port", "timeout", "retries"}
new_keys = {"host", "port", "region", "tls"}

changed_keys = {"placeholder"}
alias = changed_keys
original_id = id(changed_keys)

# Solution
changed_keys.clear()
changed_keys.update(old_keys)
changed_keys.symmetric_difference_update(new_keys)

assert changed_keys == {"timeout", "retries", "region", "tls"}
assert changed_keys is alias
assert id(changed_keys) == original_id

print(changed_keys)


{'region', 'timeout', 'retries', 'tls'}


## Practice C: Exact State Replacement Without Rebinding

Mutate `local` so it exactly matches `desired`, while returning the items added and removed.


In [54]:
local = {"A", "B", "C"}
desired = {"B", "C", "D"}

alias = local
before = local.copy()

# Solution
added = desired - before
removed = before - desired

local.intersection_update(desired)
local.update(desired)

assert local == desired
assert added == {"D"}
assert removed == {"A"}
assert local is alias

print("local:", local)
print("added:", added)
print("removed:", removed)


local: {'C', 'B', 'D'}
added: {'D'}
removed: {'A'}


# Summary

The most important lesson is that set update operations are not merely shorter syntax.

They are tools for managing **shared mutable state**.

Use them deliberately when you need to:

- preserve object identity,
- keep aliases synchronized,
- maintain an invariant incrementally,
- reconcile local and external state,
- apply layered inclusion and exclusion rules,
- reuse a live buffer,
- implement transactional rollback.

Always test both:

```python
assert target == expected_content
assert target is original_reference
```

when identity preservation is part of the requirement.
